# 02 - Exploratory Data Analysis

Deeper dive: feature-target correlations, mutual information, pairwise
interactions, and discriminative-feature discovery.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from src.utils import get_config, setup_logging
setup_logging()
cfg = get_config()
train = pd.read_csv(cfg.raw_data_dir / cfg.config.files.train_data)


## 1. Correlation heatmap

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(train.corr(), cmap='coolwarm', center=0, square=True, cbar=True,
            xticklabels=True, yticklabels=True,
            linewidths=0.4, linecolor='#1F2937')
plt.title('Correlation matrix - 30 features + Result', fontsize=14)
plt.tight_layout(); plt.show()


## 2. Top features correlated with Result

In [ ]:
corr_target = train.corr()['Result'].drop('Result').sort_values()
print('Top 5 negative (phishing-leaning):')
print(corr_target.head())
print()
print('Top 5 positive (legitimate-leaning):')
print(corr_target.tail())


## 3. Mutual information ranking

In [ ]:
from sklearn.feature_selection import mutual_info_classif
mi = mutual_info_classif(train[cfg.features], train[cfg.target_name],
                         random_state=42, discrete_features=True)
mi_series = pd.Series(mi, index=cfg.features).sort_values(ascending=False)
mi_series.head(15).plot(kind='barh', figsize=(8,5), color='#00B8FF')
plt.title('Top 15 features by Mutual Information')
plt.gca().invert_yaxis()
plt.tight_layout(); plt.show()


## 4. Pairplot on top 4 features (subsampled)

In [ ]:
top4 = mi_series.head(4).index.tolist()
sample = train.sample(1500, random_state=42)
sns.pairplot(sample[top4 + ['Result']], hue='Result',
             palette={-1: '#FF3B3B', 1: '#00E5A8'},
             plot_kws=dict(alpha=0.5, s=20))
plt.show()


## 5. Risk-score proxy (sum of feature values)

In [ ]:
train['risk_score'] = train[cfg.features].sum(axis=1)
fig, ax = plt.subplots(figsize=(9, 5))
train.loc[train.Result==-1, 'risk_score'].plot(
    kind='hist', bins=40, alpha=0.6, color='#FF3B3B', label='Phishing', ax=ax)
train.loc[train.Result== 1, 'risk_score'].plot(
    kind='hist', bins=40, alpha=0.6, color='#00E5A8', label='Legitimate', ax=ax)
ax.legend()
ax.set_xlabel('Sum of feature values'); ax.set_ylabel('Count')
ax.set_title('Risk-score distribution by class')
plt.tight_layout(); plt.show()


## Findings

* **`SSLfinal_State`, `URL_of_Anchor`, `web_traffic`** are consistently the
  most discriminative features by both Pearson correlation and mutual
  information - matching the published UCI literature.
* The simple **sum of feature values** ("risk score") already produces
  strongly bimodal class-conditional distributions, suggesting that linear
  combinations of the ternary features carry most of the signal.
